# Pipeline 3 — Temporal Particle Tracking End-to-End Test

This notebook processes a multi-frame sequence through the full Pipeline 3:
- Frame 0: Pipeline 1 cold-start (BestFirstSearcher)
- Frame 1+: IMU-predicted particle-guided two-pass search + meta-tile + semantic confirmation

Outputs: JSONL logs, meta-tiles, trajectory CSV, summary metrics.

In [ ]:
# Cell 1 — Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from config import config
from src.tile_utils import TileLoader, haversine_distance
from src.image_utils import load_image
from src.geometric_matcher import initialize_matcher
from src.semantic_model import load_semantic_model
from src.temporal_searcher import TemporalSearcher

config.ensure_output_dirs()
print('Setup complete')

In [ ]:
# Cell 2 — Load Video Sequence & IMU data
imu_log = pd.read_csv(config.IMU_CSV_PATH)
print(f'IMU log: {len(imu_log)} rows')

# Build frame list: match CSV timestamps to frame files
frame_dir = config.QUERY_FRAMES_DIR
frame_files = sorted(frame_dir.glob('frame_*.jpg'))
print(f'Found {len(frame_files)} frames in {frame_dir}')

# Map rounded timestamp -> frame path
frame_map = {}
for fp in frame_files:
    ts_str = fp.stem.replace('frame_', '')
    try:
        ts = float(ts_str)
        frame_map[round(ts, 3)] = fp
    except ValueError:
        continue

# Build aligned list: (csv_row_index, timestamp, frame_path)
NUM_FRAMES = 50  # Process first N frames for testing
aligned = []
for idx, row in imu_log.iterrows():
    ts_rounded = round(row['timestamp'], 3)
    if ts_rounded in frame_map:
        aligned.append((idx, row['timestamp'], frame_map[ts_rounded]))
    if len(aligned) >= NUM_FRAMES:
        break

print(f'Aligned {len(aligned)} frame-IMU pairs')
print(f'First: ts={aligned[0][1]:.3f}  Last: ts={aligned[-1][1]:.3f}')

In [ ]:
# Cell 3 — Initialize Temporal Searcher
semantic_model = load_semantic_model(config.SEMANTIC_MODEL_PATH, config.DEVICE)
matcher = initialize_matcher(config.DEVICE, config.MAX_NUM_KEYPOINTS)
tile_loader = TileLoader(
    config.REFERENCE_TILES_DIR,
    config.REFERENCE_PRED_DIR,
    zoom=config.TMS_ZOOM_LEVEL,
    x_range=(config.TILE_X_MIN, config.TILE_X_MAX),
    y_range=(config.TILE_Y_MIN, config.TILE_Y_MAX),
)
print(f'Tiles available: {len(tile_loader.list_tiles())}')

searcher = TemporalSearcher(semantic_model, matcher, tile_loader, config)
print('TemporalSearcher initialized')

In [ ]:
# Cell 4 — Process All Frames
results = []

for i, (csv_idx, ts, frame_path) in enumerate(aligned):
    row = imu_log.iloc[csv_idx]
    query_frame = load_image(frame_path)

    # Build IMU data dict
    # velocity: combine vel_n, vel_e if available, else use airspeed
    if 'vel_n' in row.index and 'vel_e' in row.index:
        vel = np.sqrt(row['vel_n']**2 + row['vel_e']**2)
    elif 'airspeed_true' in row.index:
        vel = row['airspeed_true']
    else:
        vel = 20.0

    gyro_z_dps = row.get('gyro_z', 0.0) * (180.0 / np.pi)

    imu_data = {
        'lat': row['latitude'],
        'lon': row['longitude'],
        'heading': np.degrees(row.get('heading_magnetic', row.get('heading', 0))),
        'pos_sigma': 100.0,
        'heading_sigma': 15.0,
        'velocity_mps': vel,
        'gyro_z_dps': gyro_z_dps,
    }

    result = searcher.process_frame(query_frame, imu_data, timestamp=ts)
    results.append(result)

    verified = result.get('meta_tile_verified', 'N/A')
    sem_conf = result.get('semantic_confidence', 0) or 0
    print(f'Frame {i:3d} | {result["method"]:20s} | tiles={result["tiles_tested"]:3d} | '
          f't={result["search_time"]:.2f}s | verified={verified} | sem={sem_conf:.3f}')

searcher.close()
print(f'\nProcessed {len(results)} frames')

In [ ]:
# Cell 5 — Analyze Performance (timing & tile counts)
frame0_time = results[0]['search_time']
subsequent_times = [r['search_time'] for r in results[1:]]
mean_sub = np.mean(subsequent_times) if subsequent_times else 0

print(f'Frame 0 time:           {frame0_time:.2f}s')
print(f'Subsequent mean time:   {mean_sub:.2f}s')
if mean_sub > 0:
    print(f'Speedup:                {frame0_time / mean_sub:.1f}x')

frame0_tiles = results[0]['tiles_tested']
subsequent_tiles = [r['tiles_tested'] for r in results[1:]]
mean_sub_tiles = np.mean(subsequent_tiles) if subsequent_tiles else 0

print(f'Frame 0 tiles:          {frame0_tiles}')
print(f'Subsequent mean tiles:  {mean_sub_tiles:.1f}')
if mean_sub_tiles > 0:
    print(f'Tile reduction:         {frame0_tiles / mean_sub_tiles:.1f}x')

In [ ]:
# Cell 6 — Compute Accuracy vs Ground Truth
errors = []
for i, (csv_idx, ts, _) in enumerate(aligned):
    row = imu_log.iloc[csv_idx]
    gt_lat = row['latitude']
    gt_lon = row['longitude']

    pos = results[i].get('position')
    if pos is None or pos[0] is None:
        errors.append(float('nan'))
        continue
    est_lat, est_lon = pos
    error_m = haversine_distance(est_lat, est_lon, gt_lat, gt_lon)
    errors.append(error_m)

errors_arr = np.array(errors)
valid = ~np.isnan(errors_arr)

print(f'Mean error:      {np.nanmean(errors_arr):.1f} m')
print(f'Median error:    {np.nanmedian(errors_arr):.1f} m')
print(f'95th percentile: {np.nanpercentile(errors_arr, 95):.1f} m')
for thresh in config.EVALUATION_THRESHOLDS:
    pct = (errors_arr[valid] < thresh).mean() * 100
    print(f'  < {thresh:4d}m: {pct:.1f}%')

In [ ]:
# Cell 7 — Visualize Trajectory
est_lats = [r['position'][0] for r in results if r['position'] and r['position'][0]]
est_lons = [r['position'][1] for r in results if r['position'] and r['position'][1]]
gt_lats = [imu_log.iloc[a[0]]['latitude'] for a in aligned[:len(est_lats)]]
gt_lons = [imu_log.iloc[a[0]]['longitude'] for a in aligned[:len(est_lons)]]

plt.figure(figsize=(12, 8))
plt.plot(gt_lons, gt_lats, 'g-', label='Ground Truth', linewidth=2)
plt.plot(est_lons, est_lats, 'r--', label='Estimated', linewidth=2)
if est_lons:
    plt.scatter(est_lons[0], est_lats[0], c='blue', s=100, marker='o', label='Start', zorder=5)
    plt.scatter(est_lons[-1], est_lats[-1], c='orange', s=100, marker='s', label='End', zorder=5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Pipeline 3 — Trajectory Comparison')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(str(config.TRAJECTORY_OUTPUT_DIR / 'trajectory_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Cell 8 — Error Timeline
plt.figure(figsize=(14, 6))
plt.plot(errors_arr, 'b-', linewidth=1.5)
plt.axhline(y=50, color='g', linestyle='--', label='50m threshold')
plt.axhline(y=100, color='orange', linestyle='--', label='100m threshold')
plt.xlabel('Frame Number')
plt.ylabel('Position Error (m)')
plt.title('Position Error Over Time')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(str(config.TRAJECTORY_OUTPUT_DIR / 'error_timeline.png'), dpi=150)
plt.show()

In [ ]:
# Cell 9 — Particle Spread Analysis
spreads = [r.get('particle_spread') for r in results[1:] if r.get('particle_spread') is not None]

if spreads:
    plt.figure(figsize=(14, 6))
    plt.plot(spreads, 'r-', linewidth=1.5)
    plt.axhline(y=200, color='red', linestyle='--', label='Divergence threshold')
    plt.xlabel('Frame Number (1+)')
    plt.ylabel('Particle Spread (m)')
    plt.title('Particle Filter Uncertainty Over Time')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(str(config.TRAJECTORY_OUTPUT_DIR / 'particle_spread.png'), dpi=150)
    plt.show()
else:
    print('No particle spread data available')

In [ ]:
# Cell 10 — Performance Summary & Export
verified_count = sum(1 for r in results[1:] if r.get('meta_tile_verified', False))
sem_confs = [r.get('semantic_confidence', 0) or 0 for r in results[1:]]

summary = {
    'Total frames': len(results),
    'Frame 0 time': f'{frame0_time:.2f}s',
    'Mean subsequent time': f'{mean_sub:.2f}s',
    'Speedup': f'{frame0_time / max(mean_sub, 0.001):.1f}x',
    'Mean error': f'{np.nanmean(errors_arr):.1f}m',
    'Median error': f'{np.nanmedian(errors_arr):.1f}m',
    'Success rate (<50m)': f'{(errors_arr[valid] < 50).mean() * 100:.1f}%',
    'Success rate (<100m)': f'{(errors_arr[valid] < 100).mean() * 100:.1f}%',
    'Meta-tile verified rate': f'{verified_count / max(len(results)-1, 1) * 100:.1f}%',
    'Mean semantic confidence': f'{np.mean(sem_confs):.3f}',
}

for key, val in summary.items():
    print(f'{key:35s}: {val}')

# Save trajectory
searcher.save_trajectory(config.TRAJECTORY_OUTPUT_DIR / 'trajectory.csv')

# Save summary
import json
with open(config.TRAJECTORY_OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nTrajectory saved to {config.TRAJECTORY_OUTPUT_DIR / "trajectory.csv"}')
print(f'Summary saved to {config.TRAJECTORY_OUTPUT_DIR / "summary.json"}')